# Notebook 3 — M3: Stress Estimation via Facial Expression
## Explainable Multimodal Diabetes/Metabolic Risk Framework

**This notebook:**
1. Trains the custom StressCNN on FER2013 (7-class emotion recognition)
2. Derives a continuous stress score from high-arousal negative emotions
3. Applies Grad-CAM to facial images
4. Saves risk scores to Drive for M5 fusion

**Stress score formula:**
> r_M3 = Σᵢ sᵢ × pᵢ  where s = [0.9, 0.7, 0.95, 0.0, 0.75, 0.1, 0.0] (angry, disgust, fear, happy, sad, surprise, neutral)

**Runtime:** ~45 min on Colab T4 GPU

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────────────────
!pip install -q torch torchvision pillow tqdm opencv-python scikit-learn matplotlib

import os, sys, numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/MultimodalDisease'
REPO_DIR = '/content/Multimodal_Disease'
os.environ['MMDISEASE_BASE'] = BASE_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from src.utils.data_utils import set_seed
set_seed(42)

In [ ]:
# ─── Load FER2013 DataLoaders ─────────────────────────────────────────────────
from src.utils.data_utils import load_fer_dataloaders
from src.config import M3

FER_DIR = f'{BASE_DIR}/data/fer2013'
train_loader, val_loader, test_loader = load_fer_dataloaders(
    fer_dir=FER_DIR, batch_size=M3['batch_size'], num_workers=2
)

print(f'Emotions: {M3["emotions"]}')
print(f'Stress weights: {M3["stress_weights"]}')

In [ ]:
# ─── Visualize class distribution ────────────────────────────────────────────
import glob
emotion_names = M3['emotions']
counts = [len(glob.glob(f'{FER_DIR}/train/{e}/*.jpg')) for e in emotion_names]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#C44E52', '#8172B2', '#4C72B0', '#55A868', '#DD8452', '#64B5CD', '#6BAD8E']
bars = ax.bar(emotion_names, counts, color=colors, alpha=0.85)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', fontsize=9)
ax.set_title('FER2013 — Emotion Class Distribution (Train Set)')
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/m3_emotion_distribution.png', dpi=150)
plt.show()

In [ ]:
# ─── Visualize stress weights ─────────────────────────────────────────────────
stress_wts = [M3['stress_weights'][i] for i in range(7)]

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ['#C44E52' if w > 0.5 else '#DD8452' if w > 0.2 else '#4C72B0' for w in stress_wts]
ax.bar(emotion_names, stress_wts, color=bar_colors, alpha=0.85)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Stress Weight sᵢ')
ax.set_title('Per-Emotion Stress Weights (r_M3 = Σ sᵢ × pᵢ)')
ax.axhline(0.5, color='gray', linestyle='--', lw=1, label='High stress threshold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/m3_stress_weights.png', dpi=150)
plt.show()

In [ ]:
# ─── Train StressCNN ──────────────────────────────────────────────────────────
from src.models.m3_stress import train_stress_model

M3_SAVE_PATH = f'{BASE_DIR}/saved_models/m3_stress.pth'

m3_model, m3_history = train_stress_model(
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = M3['epochs'],
    lr           = M3['lr'],
    save_path    = M3_SAVE_PATH
)

In [ ]:
# ─── Training curves ─────────────────────────────────────────────────────────
from src.utils.viz_utils import plot_training_curves

plot_training_curves(
    m3_history['train_losses'], m3_history['val_losses'],
    m3_history['train_accs'],   m3_history['val_accs'],
    module_name='M3 Stress CNN', save=True
)

In [ ]:
# ─── Evaluate + extract risk scores ──────────────────────────────────────────
from src.models.m3_stress import extract_risk_scores as m3_extract
from src.utils.eval_utils import compute_metrics, print_classification_report
from src.utils.viz_utils import plot_confusion_matrix

m3_stress_scores, m3_preds, m3_labels = m3_extract(m3_model, test_loader, device)

np.save(f'{BASE_DIR}/outputs/scores/m3_risk_scores.npy', m3_stress_scores)
np.save(f'{BASE_DIR}/outputs/scores/m3_labels.npy', m3_labels)
print(f'M3 stress scores saved. Shape: {m3_stress_scores.shape}')
print(f'Score range: [{m3_stress_scores.min():.3f}, {m3_stress_scores.max():.3f}]')

print('\n=== M3 Emotion Classification Metrics ===')
m3_metrics = compute_metrics(m3_labels, m3_preds, module_name='M3 Stress')
print_classification_report(m3_labels, m3_preds, class_names=emotion_names)

plot_confusion_matrix(m3_labels, m3_preds, emotion_names, 'M3 Stress (7-class)', save=True)

In [ ]:
# ─── Stress score distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(m3_stress_scores, bins=50, color='#4C72B0', alpha=0.75, edgecolor='white')
ax.axvline(0.33, color='#DD8452', lw=2, linestyle='--', label='Low/Moderate threshold')
ax.axvline(0.66, color='#C44E52', lw=2, linestyle='--', label='Moderate/High threshold')
ax.set_xlabel('Stress Risk Score (r_M3)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Stress Risk Scores on FER2013 Test Set')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/figures/m3_score_distribution.png', dpi=150)
plt.show()

In [ ]:
# ─── Grad-CAM on facial images ────────────────────────────────────────────────
from src.xai.gradcam import batch_gradcam
from src.utils.viz_utils import overlay_gradcam
import cv2

# Target: last conv block (conv4)
# Access the last conv in the sequential
target_layer = m3_model.conv4.block

m3_cam_results = batch_gradcam(
    model        = m3_model,
    dataloader   = test_loader,
    target_layer = target_layer,
    device       = device,
    n_samples    = 6
)

# FER2013 is grayscale — convert to RGB for overlay
for i, result in enumerate(m3_cam_results[:3]):
    img = result['image']
    if img.ndim == 2 or img.shape[2] == 1:  # grayscale
        img = np.stack([img[:,:,0]]*3, axis=2) if img.ndim == 3 else np.stack([img]*3, axis=2)
    
    heatmap_r = cv2.resize(result['heatmap'], (48, 48))
    emotion_true = emotion_names[result['label']]
    emotion_pred = emotion_names[result['pred']]
    
    overlay_gradcam(
        original_img = img,
        heatmap      = heatmap_r,
        title        = f'M3 Grad-CAM — Emotion: {emotion_true} | Pred: {emotion_pred}',
        save         = True,
        filename     = f'm3_gradcam_sample_{i}'
    )
print('M3 Grad-CAM figures saved.')

print('\nNotebook 3 complete.')
print(f'  M3 model: {BASE_DIR}/saved_models/m3_stress.pth')
print(f'  M3 scores: {BASE_DIR}/outputs/scores/m3_risk_scores.npy')